---
description: 面向初学者的 LangChain（Python）× Langfuse 集成示例手册。
category: Integrations
---

## 📚 前置知识导读

在开始学习 LangChain 与 Langfuse 的集成之前，让我们先了解一些基础概念：

### 🤖 LangChain 是什么？
- **定义**：LangChain 是一个用于构建大语言模型（LLM）应用的开发框架
- **作用**：帮助开发者将 LLM 与外部数据源、工具、数据库等连接起来
- **核心概念**：Chain（链）、Agent（智能体）、Tool（工具）、Memory（记忆）等

### 📊 Langfuse 是什么？
- **定义**：Langfuse 是一个专门为大语言模型应用设计的可观测性（Observability）平台
- **作用**：帮助开发者监控、调试和优化 LLM 应用的性能
- **核心功能**：追踪（Tracing）、评估（Evaluation）、监控（Monitoring）

### 🔗 两者如何协同工作？
```mermaid
graph LR
    A[你的应用] --> B[LangChain]
    B --> C[LLM/工具/数据]
    B --> D[Langfuse]
    D --> E[性能监控]
    D --> F[调试分析]
    D --> G[成本追踪]
```

### 🎯 学习目标
通过本教程，你将学会：
1. 如何在 LangChain 应用中集成 Langfuse
2. 如何追踪和监控 LLM 调用链
3. 如何分析应用性能和成本
4. 如何调试和优化 LLM 应用

---
**💡 提示**：如果你对 LangChain 或 Langfuse 还不太熟悉，建议先阅读官方文档了解基础概念。

# 示例手册：LangChain 集成

这是一本汇集了 Langfuse 与 LangChain（Python）集成示例的“示例手册”。

请按照[集成指南](https://langfuse.com/integrations/frameworks/langchain)将该集成添加到你的 LangChain 项目中。该集成也支持 LangChain JS。

## 环境准备

In [1]:
%pip install langfuse langchain langchain_openai langchain_community --upgrade

在 Langfuse 控制台的项目设置页获取 API Key，初始化 Langfuse 客户端，并将其设置到环境变量中。

In [ ]:
import os

# 在项目设置页面获取密钥：https://cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..." 
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..." 
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com" # 欧盟区域
# os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com" # 美国区域

# 你的 OpenAI API Key
os.environ["OPENAI_API_KEY"] = "sk-proj-.."

In [ ]:
from langfuse.langchain import CallbackHandler
 
# 初始化用于 LangChain 的 Langfuse 回调处理器（用于追踪）
langfuse_handler = CallbackHandler()

## 示例

### 顺序链（LCEL：LangChain 表达式语言）

![LangChain LCEL 的跟踪图](https://langfuse.com/images/cookbook/integration_langchain/langchain_LCEL.png)

[Langfuse 中的示例追踪](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/dbe646b2b67957d22e8780c429b2d20f?timestamp=2025-06-11T09%3A09%3A58.823Z&display=details)

In [ ]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

langfuse_handler = CallbackHandler()

prompt1 = ChatPromptTemplate.from_template("{person} 来自哪座城市？")
prompt2 = ChatPromptTemplate.from_template(
    "城市 {city} 位于哪个国家？请用 {language} 回答"
)
model = ChatOpenAI()
chain1 = prompt1 | model | StrOutputParser()
chain2 = (
    {"city": chain1, "language": itemgetter("language")}
    | prompt2
    | model
    | StrOutputParser()
)

chain2.invoke({"person": "obama", "language": "spanish"}, config={"callbacks":[langfuse_handler]})

'Barack Obama es de la ciudad de Chicago, Illinois, en los Estados Unidos.'

#### Runnable 方法

Runnable 是可以被调用、批处理、流式处理、转换并进行组合的工作单元。

下面的示例展示了如何在 Langfuse 中使用这些方法：

- invoke/ainvoke：将单个输入转换为输出。
- batch/abatch：高效地将多个输入批量转换为输出。
- stream/astream：在生成过程中以流式方式输出单个输入的结果。

In [ ]:
# 异步调用（Async Invoke）
await chain2.ainvoke({"person": "biden", "language": "german"}, config={"callbacks":[langfuse_handler]})

# 批处理（Batch）
chain2.batch([{"person": "elon musk", "language": "english"}, {"person": "mark zuckerberg", "language": "english"}], config={"callbacks":[langfuse_handler]})

# 异步批处理（Async Batch）
await chain2.abatch([{"person": "jeff bezos", "language": "english"}, {"person": "tim cook", "language": "english"}], config={"callbacks":[langfuse_handler]})

# 流式（Stream）
for chunk in chain2.stream({"person": "steve jobs", "language": "english"}, config={"callbacks":[langfuse_handler]}):
    print("流式分片:", chunk)

# 异步流式（Async Stream）
async for chunk in chain2.astream({"person": "bill gates", "language": "english"}, config={"callbacks":[langfuse_handler]}):
    print("异步流式分片:", chunk)


### 检索式问答（RetrievalQA）

![Langfuse 中的 LangChain 检索式问答跟踪](https://langfuse.com/images/cookbook/integration_langchain/langchain_qa_retrieval.png)

[Langfuse 中的示例追踪](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/84e1ac07dedbce3b2a236b6ece6950d9?timestamp=2025-06-11T09:29:10.248Z&display=details)

In [9]:
import os
os.environ["SERPAPI_API_KEY"] = "..."

In [10]:
%pip install unstructured selenium langchain-chroma --upgrade

In [ ]:
from langchain_community.document_loaders import SeleniumURLLoader
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.chains import RetrievalQA

langfuse_handler = CallbackHandler()

urls = [
    "https://raw.githubusercontent.com/langfuse/langfuse-docs/main/public/state_of_the_union.txt",
]
loader = SeleniumURLLoader(urls=urls)
llm = OpenAI()
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)
query = "总统对 Ketanji Brown Jackson 说了什么"
chain = RetrievalQA.from_chain_type(
    llm,
    retriever=docsearch.as_retriever(search_kwargs={"k": 1}),
)

chain.invoke(query, config={"callbacks":[langfuse_handler]})

{'query': 'What did the president say about Ketanji Brown Jackson',
 'result': " The president nominated her to serve on the United States Supreme Court and praised her as one of the nation's top legal minds who will continue the legacy of retiring Justice Stephen Breyer."}

### Azure OpenAI

In [ ]:
os.environ["AZURE_OPENAI_ENDPOINT"] = "<Azure OpenAI endpoint>"
os.environ["AZURE_OPENAI_API_KEY"] = "<Azure OpenAI API key>"
os.environ["OPENAI_API_TYPE"] = "azure"
os.environ["OPENAI_API_VERSION"] = "2023-09-01-preview"  # 固定为与所用 SDK/服务兼容的版本

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langfuse.langchain import CallbackHandler
 
# 初始化用于 LangChain 的 Langfuse 回调处理器（用于追踪）
langfuse_handler = CallbackHandler()

prompt = ChatPromptTemplate.from_template("{person} 来自哪座城市？")
model = AzureChatOpenAI(
    deployment_name="gpt-4o",
    model_name="gpt-4o",
)
chain = prompt | model

chain.invoke({"person": "Satya Nadella"}, config={"callbacks":[langfuse_handler]})

## 🎉 总结与下一步学习

恭喜！你已经完成了 LangChain 与 Langfuse 集成的基础学习。让我们回顾一下学到的内容：

### 📋 本教程涵盖的内容
✅ **环境准备**：安装依赖、配置 API 密钥、初始化回调处理器
✅ **基础集成**：在 LangChain 应用中使用 Langfuse 进行追踪
✅ **LCEL 示例**：顺序链的构建与执行
✅ **Runnable 方法**：同步/异步、批处理、流式处理
✅ **检索式问答**：文档加载、向量化、问答链构建
✅ **Azure OpenAI**：云服务集成示例

### 🚀 下一步学习建议

#### 1. 深入 LangChain 核心概念
- 学习 [LangChain 官方教程](https://python.langchain.com/docs/tutorials/)
- 掌握 Agent（智能体）和 Tool（工具）的使用
- 了解 Memory（记忆）和 Chain（链）的高级用法

#### 2. 探索 Langfuse 高级功能
- 学习如何创建自定义评估指标
- 掌握成本分析和性能优化
- 了解团队协作和项目管理功能

#### 3. 实战项目练习
- 构建一个带有多步骤推理的聊天机器人
- 创建一个支持文档检索的问答系统
- 开发一个多模态（文本+图像）应用

#### 4. 生产环境部署
- 学习 Docker 容器化部署
- 了解 Kubernetes 集群管理
- 掌握监控和日志管理最佳实践

### 🔗 有用的资源链接
- [LangChain 官方文档](https://python.langchain.com/)
- [Langfuse 官方文档](https://langfuse.com/docs)
- [LangChain 社区](https://github.com/langchain-ai/langchain)
- [Langfuse 社区](https://github.com/langfuse/langfuse)

### 💡 实践建议
1. **从简单开始**：先构建基础的问答链，再逐步增加复杂度
2. **重视监控**：始终使用 Langfuse 追踪你的应用，及时发现和解决问题
3. **迭代优化**：基于 Langfuse 的分析结果，持续优化提示词和模型参数
4. **成本控制**：监控 API 调用成本，选择性价比最高的模型组合

---
**🎯 记住**：大模型应用开发是一个迭代的过程，通过 Langfuse 的监控和分析，你可以持续改进你的应用！

祝你学习愉快，开发顺利！ 🚀